# C10 prereg42：訓練配方短跑 V3b（Colab；PLAN 修訂 3）

**不用改任何程式**：執行階段選 A100（或其他 GPU），按「執行階段 → 全部執行」。

1. 掛上雲端硬碟，核對 `我的雲端硬碟/C10/p42/C10_p42_kit.zip` 的 SHA-256，解開並逐檔核對。
2. （1b 格覆蓋 3 個新版程式檔）訓練 V3b 40,000 步（約 3 小時；每 8,000 步存到 `C10/p42/V3b/`；斷線後重按「全部執行」會接著跑）。
3. 算 D 玩家的向量（raw、EMA）存到 `C10/p42/emb/`。
4. 打包 `C10/p42/C10_p42_V3b_results.zip`，下載回傳。

任何一格出現紅字或 `STOP`，停下來，截最後 20 行。

In [ ]:
# 0. 把 Drive 上 C10/p42/kit_parts/ 的 66 份接回 C10_p42_kit.zip（已存在且雜湊對就略過）
import hashlib, os, glob
from google.colab import drive
drive.mount('/content/drive')
D = '/content/drive/MyDrive/C10/p42'
KIT = os.path.join(D, 'C10_p42_kit.zip')
SHA = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306'
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()
if not (os.path.exists(KIT) and sha(KIT) == SHA):
    parts = sorted(glob.glob(os.path.join(D, 'kit_parts', 'kit.part*')))
    assert len(parts) == 66, 'STOP: kit_parts 只有 %d 份' % len(parts)
    tmp = '/content/C10_p42_kit.zip'
    with open(tmp, 'wb') as o:
        for p in parts:
            with open(p, 'rb') as f:
                o.write(f.read())
    assert sha(tmp) == SHA, 'STOP: 接回的 zip 雜湊不對'
    import shutil; shutil.copyfile(tmp, KIT)
    assert sha(KIT) == SHA, 'STOP: 複製到 Drive 後雜湊不對'
print('kit OK', SHA[:12])


In [ ]:
# 1. 雲端硬碟、GPU、套件（不用改）
import hashlib, json, os, subprocess, sys, zipfile
from google.colab import drive
drive.mount('/content/drive')
DRIVE, WORK, VARIANT = '/content/drive/MyDrive/C10/p42', '/content/c10', 'V3b'
KIT_SHA, FILES = '6f5c93650375db2d13a3182d5c2808902b679686975438486a0342e5ad5a7306', {"data/prereg06/feats/meta.npz": "b1b5def7312dc6321ab08dd7e74e9b9069c49e9efb8caf7133b9869bf7338df5", "data/prereg36/cache/cache.json": "764a61937e60ab660972ecbe8a585e4205bd27a132ce74a64f4cc7390533c48a", "data/prereg36/cache/codes.npy": "be5831abbccf22be3c96725082939638e6607f88ad45b1ed2164db33060c0303", "data/prereg36/cache/meta.npz": "fa8bf685be64aa100746238caeaa87f917d6adc194d2026d83c7865f61f834d3", "data/prereg36/cache/off.npy": "7c352279359c5035c1693b78a783d534c8b6ac1e7f870e56655b4b764fb2eaf8", "results/prereg36/explore/p36_embed.py": "2f2e732d1309e25e0b70e370583c07b74dd4c692c6490db8fe1b533ecda32c21", "results/prereg36/explore/p36_lib.py": "8df27b37a6b3de786b54af25908d3d91ffe4c030dceb7350d9dd130a14ce241f", "results/prereg36/explore/p36_train.py": "8abc5ef22eba35b70be06c624f7071218d0a4266affe5494d1207f8819271211", "results/prereg36/explore/split_record.json": "2c847257139cd73b9a1808c003ba30b86bcd59eb6860edef8043afc82535e7e3", "results/prereg42/explore/p42_embed.py": "23b60a6f71d1aad5881acd10c87ef2c31a9ed81e6ac2d369c3d29615452b69fd", "results/prereg42/explore/p42_lib.py": "b5045f8b8222e1c3e6891675eb4b3296fe4254b974421c71853ea8a43cea6eea", "results/prereg42/explore/p42_train.py": "ec5d7baf7bf16753cd9d0fb7369e3ff4cb5a9a271d8ed28cf264691bb2359c80", "tools/c10data.py": "2906d97fe41171dc955288d0a6e695fcef905417d18d94cb98f146aac74c6495"}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''):
            h.update(b)
    return h.hexdigest()
print(subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], capture_output=True, text=True).stdout)
tv = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available())'], capture_output=True, text=True).stdout.strip()
print('torch', tv)   # in a subprocess: the notebook itself never holds the GPU (p42_train refuses to share it)
assert tv.endswith('True'), 'STOP: 沒有 GPU，請在「變更執行階段類型」選 GPU'
kit = os.path.join(DRIVE, 'C10_p42_kit.zip')
assert os.path.exists(kit), 'STOP: 找不到 ' + kit
ok_flag = os.path.join(WORK, '.kit_ok')
if not (os.path.exists(ok_flag) and open(ok_flag).read() == KIT_SHA):
    assert sha(kit) == KIT_SHA, 'STOP: 套件檔的 SHA-256 不對'
    with zipfile.ZipFile(kit) as z:
        z.extractall(WORK)
    for rel, h in FILES.items():
        assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
    open(ok_flag, 'w').write(KIT_SHA)
print('kit ok', len(FILES), 'files')
os.makedirs(os.path.join(DRIVE, VARIANT), exist_ok=True)

In [ ]:
# 1b. V3b 的新版程式（PLAN 修訂 3）：覆蓋套件裡的 p42_lib.py、p42_train.py、p42_embed.py，逐檔核雜湊（不用改）
import base64
OVERLAY = {"results/prereg42/explore/p42_embed.py": ["ea926c43bd2fd7bd5de38e14c9b5481af36de435341e82571c0797c70606b400", "IiIicHJlcmVnNDIgZXhwbG9yZTogcGVyLWdhbWUgdmVjdG9ycyBvZiBhIGNoZWNrcG9pbnQgd2l0aCB0aGUgZW1iZWQtdGltZSBvcHRpb25zIG9mIFBMQU4ubWQgMiAocmF3IC8gRU1BIHdlaWdodHMsClRUQS04LCBWMydzIHBvb2xpbmcgbW9kZSkuIEFsc28gZW1iZWRzIHByZXJlZzM2IC8gcHJlcmVnMzcgY2hlY2twb2ludHMgKHRoZSBiYXNlIHN5c3RlbSdzIENOTiwgc3BlYyBwMzYpLgoKICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNDIvZXhwbG9yZS9wNDJfZW1iZWQucHkgLS1ja3B0IGRhdGEvcHJlcmVnNDIvVjIvY2twdC9zdGVwXzQwMDAwLnB0IC0td2VpZ2h0cyBlbWEgLS1wbGF5ZXJzIEQKICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNDIvZXhwbG9yZS9wNDJfZW1iZWQucHkgLS1ja3B0IGRhdGEvcHJlcmVnMzYvY2twdC9zdGVwXzQwMDAwLnB0IC0tdHRhIDggLS1wbGF5ZXJzIEQKICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNDIvZXhwbG9yZS9wNDJfZW1iZWQucHkgLS1ja3B0IDxjKj4gW3NhbWUgb3B0aW9uc10gLS1wbGF5ZXJzIEUgICAgICAgICAgIyBhZnRlciB0dW5lLmpzb24KICAgIEdQVVBZIHJlc3VsdHMvcHJlcmVnNDIvZXhwbG9yZS9wNDJfZW1iZWQucHkgLS1ja3B0IDxjKj4gW3NhbWUgb3B0aW9uc10gLS1zb3VyY2UgcG9vbCAtLXNyYyA8dGVzdHMgZGlyPiAtLXBvb2wgcHVibGljCgpPbmUgZ2FtZSA9IHRoZSBwb29sZWQgdmVjdG9yIG9mIEFMTCB0aGUgdGFyZ2V0J3Mgc2FtcGxlIG1vdmVzIChubyBzdWJzYW1wbGluZyk7IHdpdGggLS10dGEgOCB0aGUgbWVhbiBvZiB0aGUgOCBib2FyZApzeW1tZXRyaWVzJyBnYW1lIHZlY3RvcnMuIE1lYW4gcG9vbGluZyAoVjEgLyBWMiAvIFY0IC8gcDM2KSA9IG1lYW4gb2YgdGhlIHBlci1tb3ZlIHZlY3RvcnMsIGFzIHAzNl9lbWJlZC5weS4gVjM6CidmdWxsJyA9IHRoZSB0cmFuc2Zvcm1lciBwb29saW5nIG9mIGFsbCBtb3ZlcyBhdCBvbmNlOyAnY2h1bmsyNCcgPSB0aGUgbW92ZXMgZGVhbHQgcm91bmQtcm9iaW4gaW50byB+MjQtbW92ZSBjaHVua3MsCmVhY2ggY2h1bmsgcG9vbGVkIGFuZCBMMi1ub3JtYWxpc2VkLCB0aGUgY2h1bmsgdmVjdG9ycyBhdmVyYWdlZC4gR2FtZXMgd2l0aG91dCBhIHNhbXBsZSBnZXQgY291bnQgMCBhbmQgemVyb3MuCgpPdXRwdXQgZGF0YS9wcmVyZWc0Mi9lbWIvPHRhZz4vc3RvcmVfPER8RT4ubnB6IG9yIHBvb2xfPHB1YmxpY3xwcml2YXRlPi5ucHogaW4gcDM2X2VtYmVkLnB5J3MgbGF5b3V0IChlbWIsIGNvdW50LApzdG9yZV9yb3csIGNrcHRfc2hhMjU2LCBja3B0ID0gPHRhZz4sIGNrcHRfc3RlcCkgcGx1cyBlbWJlZF9jZmcgKEpTT04pLiB0YWcgPSA8cnVuPl88Y2hlY2twb2ludCBzdGVtPl88d2VpZ2h0cz5fdHRhPG4+CltfPG1vZGU+XSwgcnVuID0gdGhlIHZhcmlhbnQgKFYxLi5WNCkgb3IgJ2Jhc2UnICsgdGhlIGZpcnN0IDggaGV4IG9mIHRoZSBjaGVja3BvaW50J3MgU0hBLTI1NiBmb3IgYSBwcmVyZWczNi1mb3JtYXQKY2hlY2twb2ludC4gUmVzdW1hYmxlIChwYXJ0cyBpbiA8b3V0cHV0Pi5wYXJ0cy8pLiAtLWNvbXBhcmUgPG5wej46IHJlY29yZCB0aGUgbGFyZ2VzdCBkaWZmZXJlbmNlIHRvIGFub3RoZXIgZmlsZSBvZgp0aGUgc2FtZSBnYW1lcyAodGhlIHNlYW0gJ3A0Ml9lbWJlZCAtLXR0YSAxJyA9PSBwMzZfZW1iZWQgb24gYSBwcmVyZWczNiBjaGVja3BvaW50KS4KIiIiCmltcG9ydCBhcmdwYXJzZQppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHN5cwppbXBvcnQgdGltZQoKaW1wb3J0IG51bXB5IGFzIG5wCgpIRVJFID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpCmlmIEhFUkUgbm90IGluIHN5cy5wYXRoOgogICAgc3lzLnBhdGguaW5zZXJ0KDAsIEhFUkUpCmltcG9ydCBwNDJfbGliIGFzIFYgICMgbm9xYTogRTQwMgoKTCA9IFYuTApDSFVOSyA9IDE2ClBBUlRfSVRFTVMgPSAxMjgKTUFYX1BPUyA9IDQwOTYKCgpjbGFzcyBHYW1lU2V0OgogICAgIiIiSXRlbSBpID0gZ2FtZXMgW2kgKiBDSFVOSywgKGkgKyAxKSAqIENIVU5LKTogeCB1aW50OCwgbG9jYWwgZ2FtZSBpbmRleCBhbmQgbW92ZSBudW1iZXIgcGVyIHBvc2l0aW9uLiIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBjb2Rlcywgc3RhcnQsIGVuZCwgaXNfd2hpdGUpOgogICAgICAgIHNlbGYuY29kZXMsIHNlbGYuc3RhcnQsIHNlbGYuZW5kLCBzZWxmLmlzX3doaXRlID0gY29kZXMsIHN0YXJ0LCBlbmQsIGlzX3doaXRlCiAgICAgICAgc2VsZi5uID0gbGVuKGlzX3doaXRlKQoKICAgIGRlZiBfX2xlbl9fKHNlbGYpOgogICAgICAgIHJldHVybiAoc2VsZi5uICsgQ0hVTksgLSAxKSAvLyBDSFVOSwoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpKToKICAgICAgICBpbXBvcnQgdG9yY2gKICAgICAgICBpZiBpc2luc3RhbmNlKHNlbGYuY29kZXMsIHN0cik6CiAgICAgICAgICAgIHNlbGYuY29kZXMgPSBucC5sb2FkKHNlbGYuY29kZXMsIG1tYXBfbW9kZT0iciIpCiAgICAgICAgeHMsIHNlZywgdHMgPSBbXSwgW10sIFtdCiAgICAgICAgZm9yIGosIGcgaW4gZW51bWVyYXRlKHJhbmdlKGkgKiBDSFVOSywgbWluKHNlbGYubiwgKGkgKyAxKSAqIENIVU5LKSkpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICB4LCB0ID0gTC5wbGFuZXNfZm9yX2NvZGVzKG5wLmFzYXJyYXkoc2VsZi5jb2Rlc1tzZWxmLnN0YXJ0W2ddOnNlbGYuZW5kW2ddXSksCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJ3IiBpZiBzZWxmLmlzX3doaXRlW2ddIGVsc2UgImIiKQogICAgICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIGlmIGxlbih4KToKICAgICAgICAgICAgICAgIHhzLmFwcGVuZCh4KQogICAgICAgICAgICAgICAgc2VnLmFwcGVuZChucC5mdWxsKGxlbih4KSwgaiwgbnAuaW50NjQpKQogICAgICAgICAgICAgICAgdHMuYXBwZW5kKG5wLmFzYXJyYXkodCwgbnAuaW50NjQpKQogICAgICAgIGlmIG5vdCB4czoKICAgICAgICAgICAgeiA9IHRvcmNoLnplcm9zKDAsIGR0eXBlPXRvcmNoLmludDY0KQogICAgICAgICAgICByZXR1cm4gaSwgdG9yY2guemVyb3MoKDAsIEwuTl9QTEFORVMsIDE5LCAxOSksIGR0eXBlPXRvcmNoLnVpbnQ4KSwgeiwgegogICAgICAgIHJldHVybiBpLCB0b3JjaC5mcm9tX251bXB5KG5wLmNvbmNhdGVuYXRlKHhzKSksIHRvcmNoLmZyb21fbnVtcHkobnAuY29uY2F0ZW5hdGUoc2VnKSksIFwKICAgICAgICAgICAgdG9yY2guZnJvbV9udW1weShucC5jb25jYXRlbmF0ZSh0cykpCgoKZGVmIGxvYWRfbmV0KGNrcHRfcGF0aCwgd2VpZ2h0cywgZGV2KToKICAgICIiIihFbWJlZE5ldCBpbiBldmFsIG1vZGUsIHNwZWMsIHJ1biBuYW1lLCBzdGVwKSBvZiBhIHByZXJlZzQyIG9yIGEgcHJlcmVnMzYtZm9ybWF0IGNoZWNrcG9pbnQuIiIiCiAgICBpbXBvcnQgdG9yY2gKICAgIGNrID0gdG9yY2gubG9hZChja3B0X3BhdGgsIG1hcF9sb2NhdGlvbj0iY3B1Iiwgd2VpZ2h0c19vbmx5PUZhbHNlKQogICAgY2ZnID0gY2suZ2V0KCJjb25maWciLCB7fSkKICAgIGlmICJ2YXJpYW50IiBpbiBjZmc6CiAgICAgICAgc3BlYyA9IFYuc3BlY19vZihjZmdbInZhcmlhbnQiXSkKICAgICAgICBpZiBjZmdbInNwZWMiXSAhPSBzcGVjOgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJ0aGUgY2hlY2twb2ludCdzIHNwZWMgZGlmZmVycyBmcm9tIHA0Ml9saWIuVkFSSUFOVFNbJXNdIiAlIGNmZ1sidmFyaWFudCJdKQogICAgICAgIG5ldCwgX2ggPSBWLmJ1aWxkKHNwZWMpCiAgICAgICAgaWYgd2VpZ2h0cyA9PSAiZW1hIjoKICAgICAgICAgICAgbmV0LmxvYWRfc3RhdGVfZGljdChja1siZW1hIl0pCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbmV0LmxvYWRfc3RhdGVfZGljdChja1sibmV0Il0pCiAgICAgICAgcnVuID0gY2ZnWyJ2YXJpYW50Il0KICAgIGVsc2U6CiAgICAgICAgaWYgd2VpZ2h0cyAhPSAicmF3IjoKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgiYSBwcmVyZWczNi1mb3JtYXQgY2hlY2twb2ludCBoYXMgbm8gRU1BIHdlaWdodHMiKQogICAgICAgIHNwZWMgPSBWLnNwZWNfb2YoInAzNiIpCiAgICAgICAgbmV0LCBfaCA9IFYuYnVpbGQoc3BlYykKICAgICAgICBuZXQudHJ1bmsubG9hZF9zdGF0ZV9kaWN0KGNrWyJuZXQiXSkKICAgICAgICBydW4gPSAiYmFzZSIgKyBMLnNoYTI1Nl9maWxlKGNrcHRfcGF0aClbOjhdCiAgICByZXR1cm4gbmV0LnRvKGRldikuZXZhbCgpLCBzcGVjLCBydW4sIGludChja1sic3RlcCJdKQoKCmRlZiBnYW1lX3ZlY3RvcnMobmV0LCBzcGVjLCB4LCBzZWcsIHQsIG5fZ2FtZXMsIHR0YSwgbW9kZSwgZGV2LCBjdWRhKToKICAgICIiIihuX2dhbWVzLCAyNTYpIGZsb2F0MzIgdmVjdG9ycyBvZiBvbmUgbG9hZGVyIGl0ZW0gYW5kIHRoZSBudW1iZXIgb2YgcG9zaXRpb25zIHBlciBnYW1lLiIiIgogICAgaW1wb3J0IHRvcmNoCiAgICBpbXBvcnQgdG9yY2gubm4uZnVuY3Rpb25hbCBhcyBGCiAgICBzeW1zID0gcmFuZ2UoOCkgaWYgdHRhID09IDggZWxzZSByYW5nZSgxKQogICAgYWNjID0gdG9yY2guemVyb3Mobl9nYW1lcywgTC5FTUJfRElNLCBkdHlwZT10b3JjaC5mbG9hdDMyLCBkZXZpY2U9ZGV2KQogICAgc2VnX2QsIHRfZCA9IHNlZy50byhkZXYpLCB0LnRvKGRldikKICAgIGNudCA9IHRvcmNoLmJpbmNvdW50KHNlZ19kLCBtaW5sZW5ndGg9bl9nYW1lcykKICAgIGlmIGxlbihzZWcpID09IDA6CiAgICAgICAgcmV0dXJuIGFjYywgY250CiAgICBpZiBtb2RlID09ICJjaHVuazI0IjoKICAgICAgICBjaWQsIG93bmVyLCBuY2ggPSBWLmNodW5rX3NlZ21lbnRzKHNlZy5udW1weSgpLCBuX2dhbWVzKQogICAgICAgIGNpZCwgb3duZXIgPSB0b3JjaC5mcm9tX251bXB5KGNpZCkudG8oZGV2KSwgdG9yY2guZnJvbV9udW1weShvd25lcikudG8oZGV2KQogICAgZm9yIHMgaW4gc3ltczoKICAgICAgICBlcyA9IFtdCiAgICAgICAgZm9yIGIgaW4gcmFuZ2UoMCwgbGVuKHNlZyksIE1BWF9QT1MpOgogICAgICAgICAgICB4YiA9IFYuc3ltX3QoeFtiOmIgKyBNQVhfUE9TXS50byhkZXYpLCBzKS5mbG9hdCgpCiAgICAgICAgICAgIHdpdGggdG9yY2guYXV0b2Nhc3QoImN1ZGEiLCBkdHlwZT10b3JjaC5mbG9hdDE2LCBlbmFibGVkPWN1ZGEpOgogICAgICAgICAgICAgICAgZXMuYXBwZW5kKG5ldC50cnVuayh4YikuZmxvYXQoKSkKICAgICAgICBlID0gdG9yY2guY2F0KGVzKQogICAgICAgIGlmIG5vdCBWLmlzX3R4KHNwZWMpOgogICAgICAgICAgICB2ID0gTC5zZWdtZW50X21lYW4oZSwgc2VnX2QsIG5fZ2FtZXMpCiAgICAgICAgZWxpZiBtb2RlID09ICJjaHVuazI0IjoKICAgICAgICAgICAgdmMgPSBGLm5vcm1hbGl6ZShuZXQucG9vbChlLCBjaWQsIG5jaCwgdF9kKS5mbG9hdCgpLCBkaW09MSkKICAgICAgICAgICAgdiA9IEwuc2VnbWVudF9tZWFuKHZjLCBvd25lciwgbl9nYW1lcykKICAgICAgICBlbHNlOgogICAgICAgICAgICB2ID0gbmV0LnBvb2woZSwgc2VnX2QsIG5fZ2FtZXMsIHRfZCkuZmxvYXQoKQogICAgICAgIGFjYyArPSB2CiAgICBhY2MgLz0gbGVuKHN5bXMpCiAgICBhY2NbY250ID09IDBdID0gMC4wCiAgICByZXR1cm4gYWNjLCBjbnQKCgpkZWYgbWFpbihhcmd2PU5vbmUpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tY2twdCIsIHJlcXVpcmVkPVRydWUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0td2VpZ2h0cyIsIGRlZmF1bHQ9InJhdyIsIGNob2ljZXM9WyJyYXciLCAiZW1hIl0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdHRhIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MSwgY2hvaWNlcz1bMSwgOF0pCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZSIsIGRlZmF1bHQ9Tm9uZSwgY2hvaWNlcz1bImZ1bGwiLCAiY2h1bmsyNCJdLCBoZWxwPSJWMyAvIFYzYiBvbmx5IChyZXF1aXJlZCB0aGVyZSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNvdXJjZSIsIGRlZmF1bHQ9InN0b3JlIiwgY2hvaWNlcz1bInN0b3JlIiwgInBvb2wiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1wbGF5ZXJzIiwgY2hvaWNlcz1bIkQiLCAiRSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNyYyIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcG9vbCIsIGNob2ljZXM9WyJwdWJsaWMiLCAicHJpdmF0ZSJdKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdvcmtlcnMiLCB0eXBlPWludCwgZGVmYXVsdD00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWRldmljZSIsIGRlZmF1bHQ9ImN1ZGEiLCBjaG9pY2VzPVsiY3VkYSIsICJjcHUiXSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1lbWItcm9vdCIsIGRlZmF1bHQ9Vi5FTUIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbGltaXQiLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lLCBoZWxwPSJzbW9rZSAvIHNlYW0gY2hlY2tzIG9ubHk6IHRoZSBmaXJzdCBOIGdhbWVzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb21wYXJlIiwgZGVmYXVsdD1Ob25lLCBoZWxwPSJhbiBucHogb2YgdGhlIHNhbWUgZ2FtZXM6IHJlY29yZCB0aGUgbGFyZ2VzdCBkaWZmZXJlbmNlIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKGFyZ3YpCiAgICB0MCA9IHRpbWUudGltZSgpCiAgICBpbXBvcnQgdG9yY2gKICAgIGZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgogICAgY3VkYSA9IGEuZGV2aWNlID09ICJjdWRhIgogICAgZGV2ID0gdG9yY2guZGV2aWNlKGEuZGV2aWNlKQogICAgaWYgbm90IGN1ZGE6CiAgICAgICAgdG9yY2guc2V0X251bV90aHJlYWRzKDIpCiAgICBuZXQsIHNwZWMsIHJ1biwgc3RlcCA9IGxvYWRfbmV0KGEuY2twdCwgYS53ZWlnaHRzLCBkZXYpCiAgICBpZiBWLmlzX3R4KHNwZWMpICE9IChhLm1vZGUgaXMgbm90IE5vbmUpOgogICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIi0tbW9kZSBmdWxsfGNodW5rMjQgaXMgcmVxdWlyZWQgZm9yIFYzIC8gVjNiIGFuZCBvbmx5IGZvciB0aGVtIikKICAgIHN0ZW0gPSBvcy5wYXRoLnNwbGl0ZXh0KG9zLnBhdGguYmFzZW5hbWUoYS5ja3B0KSlbMF0KICAgIHRhZyA9ICIlc18lc18lc190dGElZCIgJSAocnVuLCBzdGVtLCBhLndlaWdodHMsIGEudHRhKSArICgiXyVzIiAlIGEubW9kZSBpZiBhLm1vZGUgZWxzZSAiIikKICAgIG91dF9kaXIgPSBvcy5wYXRoLmpvaW4oYS5lbWJfcm9vdCwgdGFnKQogICAgZXh0cmEgPSB7fQogICAgaWYgYS5zb3VyY2UgPT0gInN0b3JlIjoKICAgICAgICBpZiBub3QgYS5wbGF5ZXJzOgogICAgICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCItLXNvdXJjZSBzdG9yZSBuZWVkcyAtLXBsYXllcnMgRCBvciBFIikKICAgICAgICBpbXBvcnQgcDM2X2VtYmVkCiAgICAgICAgY2FjaGUgPSBMLkNhY2hlKCkKICAgICAgICBFLCBELCBfVCwgX3BlciA9IEwuc3BsaXRzKGNhY2hlKQogICAgICAgIHdobyA9IHsiRCI6IEQsICJFIjogRX1bYS5wbGF5ZXJzXQogICAgICAgIHJvd3MsIGNvZGVzLCBzdGFydCwgZW5kLCBpc193aGl0ZSA9IHAzNl9lbWJlZC5zdG9yZV9nYW1lcyh3aG8sIGNhY2hlKQogICAgICAgIG5hbWUgPSAic3RvcmVfJXMiICUgYS5wbGF5ZXJzCiAgICAgICAgZXh0cmEgPSB7InN0b3JlX3JvdyI6IHJvd3MsICJwbGF5ZXJzX3NoYTI1Nl9zb3J0ZWRfaWRzIjogTC5zaGEyNTZfaWRzKHdobyl9CiAgICBlbHNlOgogICAgICAgIGlmIG5vdCAoYS5zcmMgYW5kIGEucG9vbCk6CiAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIi0tc291cmNlIHBvb2wgbmVlZHMgLS1zcmMgYW5kIC0tcG9vbCIpCiAgICAgICAgaW1wb3J0IHAzNl9lbWJlZAogICAgICAgIGNvZGVzLCBzdGFydCwgZW5kLCBpc193aGl0ZSA9IHAzNl9lbWJlZC5wb29sX2dhbWVzKGEuc3JjLCBhLnBvb2wpCiAgICAgICAgbmFtZSA9ICJwb29sXyVzIiAlIGEucG9vbAogICAgaWYgYS5saW1pdDoKICAgICAgICBpc193aGl0ZSwgc3RhcnQsIGVuZCA9IGlzX3doaXRlWzphLmxpbWl0XSwgc3RhcnRbOmEubGltaXRdLCBlbmRbOmEubGltaXRdCiAgICAgICAgZXh0cmEgPSB7azogKHZbOmEubGltaXRdIGlmIGsgPT0gInN0b3JlX3JvdyIgZWxzZSB2KSBmb3IgaywgdiBpbiBleHRyYS5pdGVtcygpfQogICAgICAgIG5hbWUgKz0gIl9saW1pdCVkIiAlIGEubGltaXQKICAgIG4gPSBsZW4oaXNfd2hpdGUpCiAgICBvdXQgPSBvcy5wYXRoLmpvaW4ob3V0X2RpciwgbmFtZSArICIubnB6IikKICAgIGlmIG9zLnBhdGguZXhpc3RzKG91dCk6CiAgICAgICAgcHJpbnQoImV4aXN0czogJXMiICUgb3V0KQogICAgICAgIHJldHVybiAwCiAgICBwYXJ0c19kaXIgPSBvdXQgKyAiLnBhcnRzIgogICAgb3MubWFrZWRpcnMocGFydHNfZGlyLCBleGlzdF9vaz1UcnVlKQogICAgY2tfc2hhID0gTC5zaGEyNTZfZmlsZShhLmNrcHQpCiAgICBlbWJlZF9jZmcgPSB7ImNrcHRfc2hhMjU2IjogY2tfc2hhLCAid2VpZ2h0cyI6IGEud2VpZ2h0cywgInR0YSI6IGEudHRhLCAibW9kZSI6IGEubW9kZSwgInNwZWMiOiBzcGVjLAogICAgICAgICAgICAgICAgICJnYW1lcyI6IGludChuKSwgIm1vdmVzIjogaW50KChucC5hc2FycmF5KGVuZCkgLSBucC5hc2FycmF5KHN0YXJ0KSkuc3VtKCkpLCAiY2h1bmsiOiBDSFVOSywKICAgICAgICAgICAgICAgICAicGFydF9pdGVtcyI6IFBBUlRfSVRFTVN9CiAgICBtZXRhX3BhdGggPSBvcy5wYXRoLmpvaW4ocGFydHNfZGlyLCAibWV0YS5qc29uIikKICAgIGlmIG9zLnBhdGguZXhpc3RzKG1ldGFfcGF0aCk6CiAgICAgICAgd2l0aCBvcGVuKG1ldGFfcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgaWYganNvbi5sb2FkKGYpICE9IGVtYmVkX2NmZzoKICAgICAgICAgICAgICAgIHJhaXNlIFN5c3RlbUV4aXQoIiVzIGJlbG9uZ3MgdG8gYW5vdGhlciBjaGVja3BvaW50IC8gc2V0dGluZyAvIGdhbWUgbGlzdDsgcmVtb3ZlIGl0IGZpcnN0IiAlIHBhcnRzX2RpcikKICAgIGVsc2U6CiAgICAgICAgTC53cml0ZV9qc29uKG1ldGFfcGF0aCwgZW1iZWRfY2ZnKQogICAgZHMgPSBHYW1lU2V0KGNvZGVzLCBzdGFydCwgZW5kLCBpc193aGl0ZSkKICAgIG5faXRlbXMgPSBsZW4oZHMpCiAgICBuX3BhcnRzID0gKG5faXRlbXMgKyBQQVJUX0lURU1TIC0gMSkgLy8gUEFSVF9JVEVNUwoKICAgIGRlZiBwYXJ0X3BhdGgocCk6CiAgICAgICAgcmV0dXJuIG9zLnBhdGguam9pbihwYXJ0c19kaXIsICJwYXJ0XyUwNWQubnB6IiAlIHApCgogICAgdG9kbyA9IFtwIGZvciBwIGluIHJhbmdlKG5fcGFydHMpIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhwYXJ0X3BhdGgocCkpXQogICAgaXRlbXMgPSBbaSBmb3IgcCBpbiB0b2RvIGZvciBpIGluIHJhbmdlKHAgKiBQQVJUX0lURU1TLCBtaW4obl9pdGVtcywgKHAgKyAxKSAqIFBBUlRfSVRFTVMpKV0KICAgIGxvYWRlciA9IERhdGFMb2FkZXIoZHMsIGJhdGNoX3NpemU9Tm9uZSwgc2FtcGxlcj1pdGVtcywgbnVtX3dvcmtlcnM9YS53b3JrZXJzLCBwaW5fbWVtb3J5PWN1ZGEsCiAgICAgICAgICAgICAgICAgICAgICAgIHByZWZldGNoX2ZhY3Rvcj00IGlmIGEud29ya2VycyBlbHNlIE5vbmUpCiAgICBidWZfZSwgYnVmX2MsIG5fcG9zID0ge30sIHt9LCAwCiAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICBmb3IgaSwgeCwgc2VnLCB0IGluIGxvYWRlcjoKICAgICAgICAgICAgaSA9IGludChpKQogICAgICAgICAgICBnID0gbWluKG4sIChpICsgMSkgKiBDSFVOSykgLSBpICogQ0hVTksKICAgICAgICAgICAgdiwgYyA9IGdhbWVfdmVjdG9ycyhuZXQsIHNwZWMsIHgsIHNlZywgdCwgZywgYS50dGEsIGEubW9kZSwgZGV2LCBjdWRhKQogICAgICAgICAgICBidWZfZVtpXSA9IHYuY3B1KCkubnVtcHkoKQogICAgICAgICAgICBidWZfY1tpXSA9IGMuY3B1KCkubnVtcHkoKS5hc3R5cGUobnAuaW50MzIpCiAgICAgICAgICAgIG5fcG9zICs9IGxlbihzZWcpCiAgICAgICAgICAgIHAgPSBpIC8vIFBBUlRfSVRFTVMKICAgICAgICAgICAgaWYgaSA9PSBtaW4obl9pdGVtcywgKHAgKyAxKSAqIFBBUlRfSVRFTVMpIC0gMToKICAgICAgICAgICAgICAgIHJyID0gcmFuZ2UocCAqIFBBUlRfSVRFTVMsIGkgKyAxKQogICAgICAgICAgICAgICAgbnAuc2F2ZXoocGFydF9wYXRoKHApICsgIi50bXAubnB6IiwgZW1iPW5wLmNvbmNhdGVuYXRlKFtidWZfZS5wb3AoaikgZm9yIGogaW4gcnJdKSwKICAgICAgICAgICAgICAgICAgICAgICAgIGNvdW50PW5wLmNvbmNhdGVuYXRlKFtidWZfYy5wb3AoaikgZm9yIGogaW4gcnJdKSkKICAgICAgICAgICAgICAgIG9zLnJlcGxhY2UocGFydF9wYXRoKHApICsgIi50bXAubnB6IiwgcGFydF9wYXRoKHApKQogICAgICAgICAgICAgICAgcHJpbnQoInBhcnQgJWQvJWQgICUuMGYgcyIgJSAocCArIDEsIG5fcGFydHMsIHRpbWUudGltZSgpIC0gdDApLCBmbHVzaD1UcnVlKQogICAgZW1iLCBjbnQgPSBbXSwgW10KICAgIGZvciBwIGluIHJhbmdlKG5fcGFydHMpOgogICAgICAgIHdpdGggbnAubG9hZChwYXJ0X3BhdGgocCkpIGFzIHo6CiAgICAgICAgICAgIGVtYi5hcHBlbmQoelsiZW1iIl0pCiAgICAgICAgICAgIGNudC5hcHBlbmQoelsiY291bnQiXSkKICAgIGVtYiA9IG5wLmNvbmNhdGVuYXRlKGVtYikuYXN0eXBlKG5wLmZsb2F0MzIpIGlmIGVtYiBlbHNlIG5wLnplcm9zKCgwLCBMLkVNQl9ESU0pLCBucC5mbG9hdDMyKQogICAgY250ID0gbnAuY29uY2F0ZW5hdGUoY250KSBpZiBjbnQgZWxzZSBucC56ZXJvcygwLCBucC5pbnQzMikKICAgIGFzc2VydCBsZW4oZW1iKSA9PSBuID09IGxlbihjbnQpCiAgICByZXAgPSB7Im91dCI6IG91dCwgInRhZyI6IHRhZywgImdhbWVzIjogaW50KG4pLCAiZ2FtZXNfd2l0aG91dF9zYW1wbGUiOiBpbnQoKGNudCA9PSAwKS5zdW0oKSksCiAgICAgICAgICAgInBvc2l0aW9uc190aGlzX3J1biI6IGludChuX3BvcyksICJzZWNvbmRzIjogcm91bmQodGltZS50aW1lKCkgLSB0MCwgMSl9CiAgICBpZiBhLmNvbXBhcmU6CiAgICAgICAgd2l0aCBucC5sb2FkKGEuY29tcGFyZSwgYWxsb3dfcGlja2xlPUZhbHNlKSBhcyB6OgogICAgICAgICAgICByZWYsIHJjID0gelsiZW1iIl1bOm5dLCB6WyJjb3VudCJdWzpuXQogICAgICAgIHJlcFsiY29tcGFyZSJdID0geyJmaWxlIjogYS5jb21wYXJlLCAiY291bnRfZXF1YWwiOiBib29sKChyYyA9PSBjbnQpLmFsbCgpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAibWF4X2Fic19kaWZmIjogZmxvYXQobnAuYWJzKHJlZiAtIGVtYikubWF4KCkpIGlmIG4gZWxzZSAwLjB9CiAgICAgICAgZW1iZWRfY2ZnWyJjb21wYXJlIl0gPSByZXBbImNvbXBhcmUiXQogICAgbnAuc2F2ZXoob3V0ICsgIi50bXAubnB6IiwgZW1iPWVtYiwgY291bnQ9Y250LCBja3B0X3NoYTI1Nj1ucC5hcnJheShja19zaGEpLCBja3B0PW5wLmFycmF5KHRhZyksCiAgICAgICAgICAgICBja3B0X3N0ZXA9bnAuYXJyYXkoc3RlcCksIGVtYmVkX2NmZz1ucC5hcnJheShqc29uLmR1bXBzKGVtYmVkX2NmZywgc29ydF9rZXlzPVRydWUpKSwgKipleHRyYSkKICAgIG9zLnJlcGxhY2Uob3V0ICsgIi50bXAubnB6Iiwgb3V0KQogICAgZm9yIHAgaW4gcmFuZ2Uobl9wYXJ0cyk6CiAgICAgICAgb3MucmVtb3ZlKHBhcnRfcGF0aChwKSkKICAgIG9zLnJlbW92ZShtZXRhX3BhdGgpCiAgICBvcy5ybWRpcihwYXJ0c19kaXIpCiAgICBwcmludChqc29uLmR1bXBzKHJlcCksIGZsdXNoPVRydWUpCiAgICByZXR1cm4gMAoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBzeXMuZXhpdChtYWluKCkpCg=="], "results/prereg42/explore/p42_lib.py": ["..."], "results/prereg42/explore/p42_train.py": ["..."]}
for rel, (h, b64) in OVERLAY.items():
    data = base64.b64decode(b64)
    assert hashlib.sha256(data).hexdigest() == h, 'STOP: 內嵌的檔案不對 ' + rel
    with open(os.path.join(WORK, rel), 'wb') as f:
        f.write(data)
    assert sha(os.path.join(WORK, rel)) == h, 'STOP: 檔案不對 ' + rel
print('overlay ok', len(OVERLAY), 'files')

In [ ]:
# 2. 訓練（不用改；斷線後重按「全部執行」會從最後的存檔接著跑）
out = os.path.join(DRIVE, VARIANT)
rec = os.path.join(out, 'train_record_%s.json' % VARIANT)
workers = str(max(2, min(12, (os.cpu_count() or 4) - 1)))
env = dict(os.environ, PYTHONDONTWRITEBYTECODE='1', PYTHONIOENCODING='utf-8')
def run(args, log):
    with open(log, 'a') as lf:
        p = subprocess.Popen([sys.executable, '-u'] + args, cwd=WORK, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='')
            lf.write(line); lf.flush()
        return p.wait()
done = os.path.exists(rec) and json.load(open(rec)).get('finished')
if not done:
    if not any(f.startswith('step_') for f in os.listdir(os.path.join(out, 'ckpt'))) if os.path.isdir(os.path.join(out, 'ckpt')) else True:
        rc = run(['results/prereg42/explore/p42_train.py', '--variant', VARIANT, '--workers', workers, '--bench', '300', '--out', out], os.path.join(out, 'bench_log.txt'))
        print('bench exit', rc)
    rc = run(['results/prereg42/explore/p42_train.py', '--variant', VARIANT, '--workers', workers, '--auto-resume', '--out', out, '--record', rec, '--min-free-ram-mb', '2048'], os.path.join(out, 'train_log.txt'))
    assert rc == 0, 'STOP: 訓練結束碼 %d（4＝記憶體不足或 STOP 檔，重按「全部執行」會接續）' % rc
print('training finished')

In [ ]:
# 3. D 玩家的向量（不用改）
ck = os.path.join(out, 'ckpt', 'step_40000.pt')
emb_root = os.path.join(DRIVE, 'emb')
jobs = [["--weights", "raw", "--mode", "full"], ["--weights", "raw", "--mode", "chunk24"], ["--weights", "ema", "--mode", "full"], ["--weights", "ema", "--mode", "chunk24"]]
for args in jobs:
    rc = run(['results/prereg42/explore/p42_embed.py', '--ckpt', ck, '--players', 'D', '--workers', workers, '--emb-root', emb_root] + args, os.path.join(out, 'embed_log.txt'))
    assert rc == 0, 'STOP: 向量結束碼 %d' % rc
print('embeddings finished')

In [ ]:
# 4. 打包回傳（不用改）
res = os.path.join(DRIVE, 'C10_p42_%s_results.zip' % VARIANT)
with zipfile.ZipFile(res + '.tmp', 'w', zipfile.ZIP_STORED) as z:
    for name in ('train_record_%s.json' % VARIANT, 'train_log.txt', 'train_log.jsonl', 'bench_log.txt', 'embed_log.txt'):
        p = os.path.join(out, name)
        if os.path.exists(p):
            z.write(p, '%s/%s' % (VARIANT, name))
    z.write(ck, '%s/ckpt/step_40000.pt' % VARIANT)
    for d in sorted(os.listdir(emb_root)):
        if d.startswith(VARIANT + '_'):
            z.write(os.path.join(emb_root, d, 'store_D.npz'), 'emb/%s/store_D.npz' % d)
os.replace(res + '.tmp', res)
drive.flush_and_unmount()
print('完成。請下載 Drive 的 C10/p42/' + os.path.basename(res) + ' 回傳給 Claude（%.0f MB）' % (os.path.getsize(res) / 1e6) if os.path.exists(res) else '完成')